In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("data/MarketLens_Customer_Lead_Dataset.csv")

In [12]:
df.head()

,Lead_ID,Customer_ID,Campaign_ID,Campaign_Name,Channel,Lead_Date,Conversion_Date,Country,City,Region,Device,Age_Group,Status,Revenue,Touch_1,Touch_2,Touch_3,Final_Touch
0,L000001,NaN,C010,Influencer Awareness,Instagram,2025-01-03,NaN,Australia,Kolkata,South,Desktop,25-34,Lost,0.00,Facebook Ads,YouTube,LinkedIn,Instagram
1,L000002,NaN,C003,Instagram Product Launch,Instagram,2025-08-17,NaN,India,Mumbai,West,Tablet,18-24,Lost,0.00,Organic Search,Organic Search,Google Ads,Instagram
2,L000003,CUST000002,C007,Google Display Remarketing,Google Ads,2025-08-17,2025-09-24,UK,Mumbai,East,Mobile,35-44,Customer,983.16,YouTube,Instagram,Facebook Ads,Google Ads
3,L000004,NaN,C016,High Value Customer Campaign,Email,2025-12-09,NaN,Germany,Chennai,South,Desktop,25-34,Lost,0.00,Google Ads,Instagram,Facebook Ads,Email
4,L000005,NaN,C016,High Value Customer Campaign,Email,2025-05-21,NaN,India,New York,East,Desktop,18-24,Lost,0.00,Organic Search,Organic Search,Organic Search,Email


In [13]:
print(df.shape)

(50000, 18)


In [10]:
df.columns

Index(['Lead_ID', 'Customer_ID', 'Campaign_ID', 'Campaign_Name', 'Channel',
       'Lead_Date', 'Conversion_Date', 'Country', 'City', 'Region', 'Device',
       'Age_Group', 'Status', 'Revenue', 'Touch_1', 'Touch_2', 'Touch_3',
       'Final_Touch'],
      dtype='str')

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Lead_ID          50000 non-null  str    
 1   Customer_ID      8165 non-null   str    
 2   Campaign_ID      50000 non-null  str    
 3   Campaign_Name    50000 non-null  str    
 4   Channel          50000 non-null  str    
 5   Lead_Date        50000 non-null  str    
 6   Conversion_Date  8165 non-null   str    
 7   Country          50000 non-null  str    
 8   City             50000 non-null  str    
 9   Region           50000 non-null  str    
 10  Device           50000 non-null  str    
 11  Age_Group        50000 non-null  str    
 12  Status           50000 non-null  str    
 13  Revenue          50000 non-null  float64
 14  Touch_1          50000 non-null  str    
 15  Touch_2          50000 non-null  str    
 16  Touch_3          50000 non-null  str    
 17  Final_Touch      50000 

In [12]:
df.describe()

,Revenue
count,50000.000000
mean,162.965100
std,414.715333
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1799.840000


In [13]:
df.isnull().sum()

Lead_ID                0
Customer_ID        41835
Campaign_ID            0
Campaign_Name          0
Channel                0
Lead_Date              0
Conversion_Date    41835
Country                0
City                   0
Region                 0
Device                 0
Age_Group              0
Status                 0
Revenue                0
Touch_1                0
Touch_2                0
Touch_3                0
Final_Touch            0
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df.nunique()

Lead_ID            50000
Customer_ID         8165
Campaign_ID           20
Campaign_Name         20
Channel                7
Lead_Date            608
Conversion_Date      648
Country                7
City                  12
Region                 5
Device                 3
Age_Group              5
Status                 3
Revenue             7961
Touch_1                7
Touch_2                7
Touch_3                7
Final_Touch            7
dtype: int64

In [16]:
df = df.drop_duplicates()

In [17]:
df["Lead_Date"] = pd.to_datetime(df["Lead_Date"])

df["Conversion_Date"] = pd.to_datetime(
    df["Conversion_Date"],
    errors="coerce"
)

In [18]:
text_columns = [
    "Campaign_Name",
    "Channel",
    "Country",
    "City",
    "Region",
    "Device",
    "Age_Group",
    "Status"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [19]:
df["Revenue"].describe()

count    50000.000000
mean       162.965100
std        414.715333
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       1799.840000
Name: Revenue, dtype: float64

In [20]:
df[df["Revenue"] < 0]

,Lead_ID,Customer_ID,Campaign_ID,Campaign_Name,Channel,Lead_Date,Conversion_Date,Country,City,Region,Device,Age_Group,Status,Revenue,Touch_1,Touch_2,Touch_3,Final_Touch


In [21]:
df["Converted"] = np.where(
    df["Status"] == "Customer", 1, 0
)

In [22]:
df["Conversion_Days"] = (
    df["Conversion_Date"] - df["Lead_Date"]
).dt.days

In [23]:
df["Lead_Month"] = df["Lead_Date"].dt.to_period("M").astype(str)

In [24]:
df["Lead_Year"] = df["Lead_Date"].dt.year

In [25]:
#How many leads?
total_leads = df["Lead_ID"].nunique()

print(total_leads)

50000


In [26]:
#How many customers?
customers = df[df["Status"] == "Customer"]["Customer_ID"].nunique()

print(customers)

8165


In [27]:
#Conversion rate
conversion_rate = customers / total_leads * 100

print(conversion_rate)

16.33


Leads by channel

In [28]:
channel_leads = df.groupby("Channel")["Lead_ID"].nunique()

print(channel_leads)

Channel
Email              7443
Facebook Ads       7380
Google Ads        12464
Instagram          7488
LinkedIn           7490
Organic Search     2588
YouTube            5147
Name: Lead_ID, dtype: int64


Customers by channel

In [29]:
channel_customers = (
    df[df["Status"] == "Customer"]
    .groupby("Channel")["Customer_ID"]
    .nunique()
)
print(channel_customers)

Channel
Email             1201
Facebook Ads      1186
Google Ads        2032
Instagram         1197
LinkedIn          1279
Organic Search     430
YouTube            840
Name: Customer_ID, dtype: int64


Revenue by channel

In [15]:
channel_revenue = (
    df.groupby("Channel")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(channel_revenue)

Channel
Google Ads        2028950.48
LinkedIn          1277317.84
Email             1198060.66
Instagram         1188569.23
Facebook Ads      1185586.69
YouTube            843562.78
Organic Search     426207.32
Name: Revenue, dtype: float64


Customer segmentation

In [16]:
#By device
df.groupby("Device")["Revenue"].sum()

Device
Desktop    2414853.02
Mobile     5058212.84
Tablet      675189.14
Name: Revenue, dtype: float64

In [17]:
#By age
df.groupby("Age_Group")["Revenue"].sum()

Age_Group
18-24    1013648.84
25-34    2803939.18
35-44    2241805.59
45-54    1364526.73
55+       724334.66
Name: Revenue, dtype: float64

In [18]:
#By region
df.groupby("Region")["Revenue"].sum()

Region
Central    1651102.46
East       1605724.72
North      1614231.35
South      1658150.62
West       1619045.85
Name: Revenue, dtype: float64

In [19]:
#By country
df.groupby("Country")["Revenue"].sum()

Country
Australia    1136717.48
Canada       1104938.42
Germany      1143518.80
India        1175314.63
UAE          1204743.17
UK           1217264.51
USA          1165757.99
Name: Revenue, dtype: float64

In [23]:
#By city
df.groupby("City")["Revenue"].sum().sort_values(
    ascending=False
)

City
Bengaluru    743034.95
Toronto      709619.99
Kolkata      701228.95
New York     690163.50
Mumbai       679370.43
Chennai      677624.71
Dubai        674635.38
Delhi        671403.88
Pune         665166.08
Hyderabad    651896.54
London       648536.50
Ahmedabad    635574.09
Name: Revenue, dtype: float64

CAMPAIGN DATASET

In [30]:
campaigns = pd.read_excel(
    "data/campaign_performance.xlsx"
)

In [31]:
campaigns.head()

,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,Leads,Qualified_Leads,Customers,Revenue
0,C001,Google Search Summer Sale,Google Ads,Search,2025-04-13,2025-07-15,66632.37,55791.79,1760908,72121,8574,3193,470,475997.44
1,C002,Meta New Customer Push,Facebook Ads,Social,2026-03-12,2026-06-09,147736.17,111827.18,2036222,125876,5514,2985,1251,1243884.35
2,C003,Instagram Product Launch,Instagram,Social,2025-09-28,2025-11-17,78012.99,69435.39,1128732,69546,7413,1854,343,114470.04
3,C004,YouTube Brand Awareness,YouTube,Video,2025-04-17,2025-06-18,131091.95,90661.35,1575472,88842,8695,3553,1465,1023101.43
4,C005,LinkedIn B2B Leads,LinkedIn,Social,2025-03-13,2025-06-26,106841.52,105948.90,198984,6811,356,137,31,27581.60


In [32]:
print(campaigns.shape)

(20, 14)


In [33]:
campaigns.info()
campaigns.describe()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Campaign_ID      20 non-null     str           
 1   Campaign_Name    20 non-null     str           
 2   Channel          20 non-null     str           
 3   Campaign_Type    20 non-null     str           
 4   Start_Date       20 non-null     datetime64[us]
 5   End_Date         20 non-null     datetime64[us]
 6   Budget           20 non-null     float64       
 7   Spend            20 non-null     float64       
 8   Impressions      20 non-null     int64         
 9   Clicks           20 non-null     int64         
 10  Leads            20 non-null     int64         
 11  Qualified_Leads  20 non-null     int64         
 12  Customers        20 non-null     int64         
 13  Revenue          20 non-null     float64       
dtypes: datetime64[us](2), float64(3), int64(5), str(4)
memo

,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,Leads,Qualified_Leads,Customers,Revenue
count,20,20,20.000000,20.000000,2.000000e+01,20.000000,20.00000,20.000000,20.000000,2.000000e+01
mean,2025-09-19 01:12:00,2025-12-12 04:48:00,81445.230500,72291.392000,1.497130e+06,75305.600000,6949.75000,3048.800000,816.550000,5.706246e+05
min,2025-01-21 00:00:00,2025-04-18 00:00:00,16790.770000,13872.630000,1.989840e+05,6811.000000,356.00000,137.000000,31.000000,2.758160e+04
25%,2025-04-16 00:00:00,2025-06-25 12:00:00,47196.497500,46997.692500,1.030586e+06,47482.250000,3625.25000,1279.500000,318.500000,1.723649e+05
50%,2025-10-17 00:00:00,2025-12-26 12:00:00,76915.195000,69873.355000,1.707646e+06,70833.500000,4223.50000,2089.000000,480.000000,3.520712e+05
75%,2026-01-28 00:00:00,2026-04-20 18:00:00,109673.882500,96323.977500,2.073077e+06,92341.000000,8604.25000,3663.500000,957.000000,5.121231e+05
max,2026-05-07 00:00:00,2026-08-08 00:00:00,147736.170000,145448.410000,2.381785e+06,197663.000000,23481.00000,11436.000000,3634.000000,3.290124e+06
std,NaN,NaN,40101.776009,35405.038614,7.372170e+05,49505.509227,6067.65464,3089.838705,986.364482,7.391530e+05


In [34]:
campaigns["CTR"] = (
    campaigns["Clicks"] /
    campaigns["Impressions"] * 100
)

In [35]:
campaigns["Lead_Conversion_Rate"] = (
    campaigns["Leads"] /
    campaigns["Clicks"] * 100
)

In [36]:
campaigns["Customer_Conversion_Rate"] = (
    campaigns["Customers"] /
    campaigns["Leads"] * 100
)

In [37]:
campaigns["CPL"] = (
    campaigns["Spend"] /
    campaigns["Leads"]
)

In [38]:
campaigns["CAC"] = (
    campaigns["Spend"] /
    campaigns["Customers"]
)

In [39]:
campaigns["ROAS"] = (
    campaigns["Revenue"] /
    campaigns["Spend"]
)

In [40]:
campaigns["ROI"] = (
    (campaigns["Revenue"] - campaigns["Spend"])
    / campaigns["Spend"]
) * 100

In [41]:
#Highest revenue
campaigns.sort_values(
    "Revenue",
    ascending=False
).head(10)

,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,...,Qualified_Leads,Customers,Revenue,CTR,Lead_Conversion_Rate,Customer_Conversion_Rate,CPL,CAC,ROAS,ROI
8,C009,Holiday Shopping Promo,Google Ads,Search,2025-08-03,2025-10-20,91043.91,86408.92,1902126,141563,...,11436,3634,3290123.75,7.442357,14.385821,17.844341,4.243011,23.777909,38.076205,3707.620498
7,C008,Meta Retargeting,Facebook Ads,Social,2026-04-12,2026-08-08,142197.24,145448.41,2380919,197663,...,10993,3330,1347470.74,8.301962,11.879310,14.181679,6.194302,43.678201,9.264252,826.425212
1,C002,Meta New Customer Push,Facebook Ads,Social,2026-03-12,2026-06-09,147736.17,111827.18,2036222,125876,...,2985,1251,1243884.35,6.181841,4.380501,22.687704,20.280591,89.390232,11.123274,1012.327388
3,C004,YouTube Brand Awareness,YouTube,Video,2025-04-17,2025-06-18,131091.95,90661.35,1575472,88842,...,3553,1465,1023101.43,5.639072,9.787038,16.848764,10.426837,61.884881,11.284869,1028.486869
5,C006,Email Retention Q3,Email,Email,2025-01-21,2025-04-18,75817.40,64172.53,1654384,89811,...,4432,490,517207.73,5.428667,15.227533,3.582919,4.692346,130.964347,8.059644,705.964374
6,C007,Google Display Remarketing,Google Ads,Display,2025-05-02,2025-06-22,16790.77,13872.63,2183643,91286,...,4737,996,510428.17,4.180445,10.584317,10.308425,1.435793,13.928343,36.793901,3579.390065
0,C001,Google Search Summer Sale,Google Ads,Search,2025-04-13,2025-07-15,66632.37,55791.79,1760908,72121,...,3193,470,475997.44,4.095671,11.888354,5.481689,6.507090,118.705936,8.531675,753.167536
16,C017,Mobile App Acquisition,Google Ads,App,2025-12-10,2026-03-11,38404.23,34452.36,1993400,58048,...,1454,519,451116.83,2.912010,6.877067,13.001002,8.630351,66.382197,13.093931,1209.393116
19,C020,Year End Mega Sale,Google Ads,Search,2026-01-21,2026-04-11,116973.79,91487.51,501279,27284,...,2294,522,382248.56,5.442877,14.741240,12.978618,22.746770,175.263429,4.178150,317.815022
18,C019,Enterprise ABM,LinkedIn,Lead Gen,2026-02-18,2026-05-20,39601.87,35823.96,1146107,55329,...,2278,944,371230.88,4.827560,7.309006,23.343225,8.858546,37.949110,10.362642,936.264221


In [42]:
#Best ROAS
campaigns.sort_values(
    "ROAS",
    ascending=False
).head(10)

,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,...,Qualified_Leads,Customers,Revenue,CTR,Lead_Conversion_Rate,Customer_Conversion_Rate,CPL,CAC,ROAS,ROI
8,C009,Holiday Shopping Promo,Google Ads,Search,2025-08-03,2025-10-20,91043.91,86408.92,1902126,141563,...,11436,3634,3290123.75,7.442357,14.385821,17.844341,4.243011,23.777909,38.076205,3707.620498
6,C007,Google Display Remarketing,Google Ads,Display,2025-05-02,2025-06-22,16790.77,13872.63,2183643,91286,...,4737,996,510428.17,4.180445,10.584317,10.308425,1.435793,13.928343,36.793901,3579.390065
16,C017,Mobile App Acquisition,Google Ads,App,2025-12-10,2026-03-11,38404.23,34452.36,1993400,58048,...,1454,519,451116.83,2.912010,6.877067,13.001002,8.630351,66.382197,13.093931,1209.393116
10,C011,Cart Recovery Email,Email,Email,2026-04-04,2026-07-01,17155.44,15286.60,1887095,59479,...,1389,337,180510.45,3.151882,6.489685,8.730570,3.960259,45.360831,11.808411,1080.841063
3,C004,YouTube Brand Awareness,YouTube,Video,2025-04-17,2025-06-18,131091.95,90661.35,1575472,88842,...,3553,1465,1023101.43,5.639072,9.787038,16.848764,10.426837,61.884881,11.284869,1028.486869
1,C002,Meta New Customer Push,Facebook Ads,Social,2026-03-12,2026-06-09,147736.17,111827.18,2036222,125876,...,2985,1251,1243884.35,6.181841,4.380501,22.687704,20.280591,89.390232,11.123274,1012.327388
18,C019,Enterprise ABM,LinkedIn,Lead Gen,2026-02-18,2026-05-20,39601.87,35823.96,1146107,55329,...,2278,944,371230.88,4.827560,7.309006,23.343225,8.858546,37.949110,10.362642,936.264221
7,C008,Meta Retargeting,Facebook Ads,Social,2026-04-12,2026-08-08,142197.24,145448.41,2380919,197663,...,10993,3330,1347470.74,8.301962,11.879310,14.181679,6.194302,43.678201,9.264252,826.425212
0,C001,Google Search Summer Sale,Google Ads,Search,2025-04-13,2025-07-15,66632.37,55791.79,1760908,72121,...,3193,470,475997.44,4.095671,11.888354,5.481689,6.507090,118.705936,8.531675,753.167536
5,C006,Email Retention Q3,Email,Email,2025-01-21,2025-04-18,75817.40,64172.53,1654384,89811,...,4432,490,517207.73,5.428667,15.227533,3.582919,4.692346,130.964347,8.059644,705.964374


In [ ]:
#Lowest CAC
campaigns.sort_values(
    "CAC",
    ascending=True
).head(10)

,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,...,Qualified_Leads,Customers,Revenue,CTR,Lead_Conversion_Rate,Customer_Conversion_Rate,CPL,CAC,ROAS,ROI
6,C007,Google Display Remarketing,Google Ads,Display,2025-05-02,2025-06-22,16790.77,13872.63,2183643,91286,...,4737,996,510428.17,4.180445,10.584317,10.308425,1.435793,13.928343,36.793901,3579.390065
8,C009,Holiday Shopping Promo,Google Ads,Search,2025-08-03,2025-10-20,91043.91,86408.92,1902126,141563,...,11436,3634,3290123.75,7.442357,14.385821,17.844341,4.243011,23.777909,38.076205,3707.620498
18,C019,Enterprise ABM,LinkedIn,Lead Gen,2026-02-18,2026-05-20,39601.87,35823.96,1146107,55329,...,2278,944,371230.88,4.827560,7.309006,23.343225,8.858546,37.949110,10.362642,936.264221
7,C008,Meta Retargeting,Facebook Ads,Social,2026-04-12,2026-08-08,142197.24,145448.41,2380919,197663,...,10993,3330,1347470.74,8.301962,11.879310,14.181679,6.194302,43.678201,9.264252,826.425212
10,C011,Cart Recovery Email,Email,Email,2026-04-04,2026-07-01,17155.44,15286.60,1887095,59479,...,1389,337,180510.45,3.151882,6.489685,8.730570,3.960259,45.360831,11.808411,1080.841063
3,C004,YouTube Brand Awareness,YouTube,Video,2025-04-17,2025-06-18,131091.95,90661.35,1575472,88842,...,3553,1465,1023101.43,5.639072,9.787038,16.848764,10.426837,61.884881,11.284869,1028.486869
16,C017,Mobile App Acquisition,Google Ads,App,2025-12-10,2026-03-11,38404.23,34452.36,1993400,58048,...,1454,519,451116.83,2.912010,6.877067,13.001002,8.630351,66.382197,13.093931,1209.393116
1,C002,Meta New Customer Push,Facebook Ads,Social,2026-03-12,2026-06-09,147736.17,111827.18,2036222,125876,...,2985,1251,1243884.35,6.181841,4.380501,22.687704,20.280591,89.390232,11.123274,1012.327388
0,C001,Google Search Summer Sale,Google Ads,Search,2025-04-13,2025-07-15,66632.37,55791.79,1760908,72121,...,3193,470,475997.44,4.095671,11.888354,5.481689,6.507090,118.705936,8.531675,753.167536
5,C006,Email Retention Q3,Email,Email,2025-01-21,2025-04-18,75817.40,64172.53,1654384,89811,...,4432,490,517207.73,5.428667,15.227533,3.582919,4.692346,130.964347,8.059644,705.964374


In [44]:
#Highest ROI
campaigns.sort_values(
    "ROI",
    ascending=False
).head(10)

,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Start_Date,End_Date,Budget,Spend,Impressions,Clicks,...,Qualified_Leads,Customers,Revenue,CTR,Lead_Conversion_Rate,Customer_Conversion_Rate,CPL,CAC,ROAS,ROI
8,C009,Holiday Shopping Promo,Google Ads,Search,2025-08-03,2025-10-20,91043.91,86408.92,1902126,141563,...,11436,3634,3290123.75,7.442357,14.385821,17.844341,4.243011,23.777909,38.076205,3707.620498
6,C007,Google Display Remarketing,Google Ads,Display,2025-05-02,2025-06-22,16790.77,13872.63,2183643,91286,...,4737,996,510428.17,4.180445,10.584317,10.308425,1.435793,13.928343,36.793901,3579.390065
16,C017,Mobile App Acquisition,Google Ads,App,2025-12-10,2026-03-11,38404.23,34452.36,1993400,58048,...,1454,519,451116.83,2.912010,6.877067,13.001002,8.630351,66.382197,13.093931,1209.393116
10,C011,Cart Recovery Email,Email,Email,2026-04-04,2026-07-01,17155.44,15286.60,1887095,59479,...,1389,337,180510.45,3.151882,6.489685,8.730570,3.960259,45.360831,11.808411,1080.841063
3,C004,YouTube Brand Awareness,YouTube,Video,2025-04-17,2025-06-18,131091.95,90661.35,1575472,88842,...,3553,1465,1023101.43,5.639072,9.787038,16.848764,10.426837,61.884881,11.284869,1028.486869
1,C002,Meta New Customer Push,Facebook Ads,Social,2026-03-12,2026-06-09,147736.17,111827.18,2036222,125876,...,2985,1251,1243884.35,6.181841,4.380501,22.687704,20.280591,89.390232,11.123274,1012.327388
18,C019,Enterprise ABM,LinkedIn,Lead Gen,2026-02-18,2026-05-20,39601.87,35823.96,1146107,55329,...,2278,944,371230.88,4.827560,7.309006,23.343225,8.858546,37.949110,10.362642,936.264221
7,C008,Meta Retargeting,Facebook Ads,Social,2026-04-12,2026-08-08,142197.24,145448.41,2380919,197663,...,10993,3330,1347470.74,8.301962,11.879310,14.181679,6.194302,43.678201,9.264252,826.425212
0,C001,Google Search Summer Sale,Google Ads,Search,2025-04-13,2025-07-15,66632.37,55791.79,1760908,72121,...,3193,470,475997.44,4.095671,11.888354,5.481689,6.507090,118.705936,8.531675,753.167536
5,C006,Email Retention Q3,Email,Email,2025-01-21,2025-04-18,75817.40,64172.53,1654384,89811,...,4432,490,517207.73,5.428667,15.227533,3.582919,4.692346,130.964347,8.059644,705.964374


Funnel analysis

In [5]:
events = pd.read_csv(
    "data/raw/marketing_events.csv"
)

In [6]:
events["Event_Type"].value_counts()

Event_Type
Impression        57966
Click             20014
Lead              12119
Qualified Lead     5916
Customer           3985
Name: count, dtype: int64

In [7]:
funnel = events.groupby(
    "Event_Type"
).size()

print(funnel)

Event_Type
Click             20014
Customer           3985
Impression        57966
Lead              12119
Qualified Lead     5916
dtype: int64


In [8]:
#conversion rate
click_to_lead = (
    funnel["Lead"] /
    funnel["Click"] * 100
)

Attribution Analysis

In [9]:
journeys = pd.read_csv(
    "data/raw/customer_journeys.csv"
)

In [10]:
journeys.head()

,Customer_ID,First_Touch,Second_Touch,Third_Touch,Last_Touch,Converted,Revenue
0,CUST000001,YouTube,Social Media,Social Media,Google Ads,0,0
1,CUST000002,LinkedIn,Google Ads,Email,Email,0,0
2,CUST000003,Email,Google Ads,Google Ads,YouTube,0,0
3,CUST000004,LinkedIn,Google Ads,Social Media,LinkedIn,0,0
4,CUST000005,LinkedIn,Email,Google Ads,LinkedIn,1,2905


In [11]:
first_touch = (
    journeys[journeys["Converted"] == 1]
    .groupby("First_Touch")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

In [12]:
last_touch = (
    journeys[journeys["Converted"] == 1]
    .groupby("Last_Touch")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

Exporting Cleaned Data

In [13]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

print("Cleaned folder ready!")

Cleaned folder ready!


In [19]:
import os

print("Current folder:")
print(os.getcwd())

Current folder:
c:\Users\ADMIN\Desktop\MarketLens_Project


In [26]:
import os

print(os.path.abspath("MarketLens_Project/data/cleaned"))

c:\Users\ADMIN\Desktop\MarketLens_Project\MarketLens_Project\data\cleaned


In [29]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

df.to_csv("../data/cleaned/clean_leads_customers.csv", index=False)
campaigns.to_csv("../data/cleaned/clean_campaigns.csv", index=False)
events.to_csv("../data/cleaned/clean_events.csv", index=False)
journeys.to_csv("../data/cleaned/clean_journeys.csv", index=False)

print("Done!")

NameError: name 'campaigns' is not defined

In [32]:
import pandas as pd
import os

# Create cleaned folder
os.makedirs("../data/cleaned", exist_ok=True)

# Load all datasets
df = pd.read_csv("../data/raw/MarketLens_Customer_Lead_Dataset.csv")

campaigns = pd.read_csv("../data/raw/campaign_performance.csv")

events = pd.read_csv("../data/raw/marketing_events.csv")

journeys = pd.read_csv("../data/raw/customer_journeys.csv")

print("All datasets loaded successfully!")
print("Customer/Lead:", df.shape)
print("Campaigns:", campaigns.shape)
print("Events:", events.shape)
print("Journeys:", journeys.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/MarketLens_Customer_Lead_Dataset.csv'

In [33]:
import os

print("Current folder:")
print(os.getcwd())

Current folder:
c:\Users\ADMIN\Desktop\MarketLens_Project


In [38]:
import pandas as pd
import os

os.makedirs("data/cleaned", exist_ok=True)

df = pd.read_csv("data/raw/MarketLens_Customer_Lead_Dataset.csv")

campaigns = pd.read_excel("data/raw/campaign_performance.xlsx")

events = pd.read_csv("data/raw/marketing_events.csv")

journeys = pd.read_csv("data/raw/customer_journeys.csv")

print(" All datasets loaded successfully!")
print("Customer/Lead:", df.shape)
print("Campaigns:", campaigns.shape)
print("Events:", events.shape)
print("Journeys:", journeys.shape)

 All datasets loaded successfully!
Customer/Lead: (50000, 18)
Campaigns: (20, 14)
Events: (100000, 7)
Journeys: (20000, 7)


In [39]:
df.to_csv("data/cleaned/clean_leads_customers.csv", index=False)
campaigns.to_excel("data/cleaned/clean_campaigns.xlsx", index=False)
events.to_csv("data/cleaned/clean_events.csv", index=False)
journeys.to_csv("data/cleaned/clean_journeys.csv", index=False)

print("✅ All 4 cleaned files created!")

✅ All 4 cleaned files created!
